# M7.1 · GPU programlama modeli ve C++/CUDA extension

GPU'nun nasıl düşündüğünü CPU'da öğrenmek. Grid/block/thread indekslerini ve tiling'in global bellek okumalarını nasıl azalttığını elle hesaplıyor, saf Python'da bir CUDA ızgarası simülatörü ve okuma sayan tiled matmul yazıyor, NumPy'da roofline modeliyle işlemleri bellek- ya da hesap-bağımlı diye sınıflandırıyor, `torch.utils.cpp_extension.load_inline` ile kendi C++ operatörünü derleyip PyTorch'a bağlıyor (GPU varsa CUDA çekirdeğiyle) ve en sonda döngü sırasının önbellek üzerindeki etkisini C++'ta ölçüyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m7-1-cuda.html](https://m-gonul.github.io/pytorch-lab/m7-1-cuda.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m7_1_cuda.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    Global = need(ns, \"Global\", \"sınıf\")\n\n    def va():\n        n = 1000\n        x, y, out = Global(range(n)), Global([2 * i for i in range(n)]), Global([None] * n)\n        need(ns, \"launch\")(need(ns, \"vec_add\"), 4, 256, x, y, out, n)\n        ensure(out.data == [3 * i for i in range(n)], \"sonuç yanlış\")\n        ensure(x.reads == n, f\"x {x.reads} kez okundu; sınır dışı thread'ler okumamalı\")\n    R.test(\"vec_add + launch (sınır kontrolü)\", va)\n    rng = random.Random(0)\n\n    def mm(fn, *extra):\n        n = 8\n        a = [rng.uniform(-1, 1) for _ in range(n * n)]; b = [rng.uniform(-1, 1) for _ in range(n * n)]\n        A, B = Global(a), Global(b)\n        C = fn(A, B, n, *extra)\n        ref = [sum(a[i * n + k] * b[k * n + j] for k in range(n)) for i in range(n) for j in range(n)]\n        close(C, ref)\n        return A.reads + B.reads\n\n    def naive():\n        r = mm(need(ns, \"matmul_naive\"))\n        ensure(r == 2 * 8 ** 3, f\"{r} okuma; 2n³ = 1024 bekleniyordu\")\n    R.test(\"matmul_naive: 2n³ okuma\", naive)\n\n    def tiled():\n        rows = [[\"naif\", str(2 * 8 ** 3)]]\n        for T in (2, 4, 8):\n            r = mm(need(ns, \"matmul_tiled\"), T)\n            ensure(r == 2 * 8 ** 3 // T, f\"T = {T}: {r} okuma; 2n³/T = {2 * 8 ** 3 // T} bekleniyordu\")\n            rows.append([f\"tile {T}\", str(r)])\n        R.table(\"n = 8, global okuma\", [\"yöntem\", \"okuma\"], rows)\n    R.test(\"matmul_tiled: 2n³/T okuma\", tiled)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"intensity/attainable döngüsüz\", lambda: (max_loops(src, 0, func=\"intensity\"), max_loops(src, 0, func=\"attainable\")))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    R.test(\"intensity\", lambda: close(need(ns, \"intensity\")([1, 10], [12, 4]), [1 / 12, 2.5]))\n\n    def att():\n        I = np.array([0.083, 1.0, 12.54, 100.0])\n        close(need(ns, \"attainable\")(19.5e12, 1.555e12, I), np.minimum(19.5e12, 1.555e12 * I))\n    R.test(\"attainable (roofline)\", att)\n\n    def oc():\n        c = need(ns, \"op_costs\")(1024)\n        ensure(tuple(c[\"add\"]) == (1024, 12 * 1024), \"add: n FLOP, 3n·4 bayt\")\n        ensure(tuple(c[\"matmul\"]) == (2 * 1024 ** 3, 12 * 1024 ** 2), \"matmul: 2n³ FLOP, 3n²·4 bayt\")\n        ensure(tuple(c[\"softmax\"]) == (5 * 1024 ** 2, 8 * 1024 ** 2), \"softmax: 5n² FLOP, 2n²·4 bayt\")\n        ridge = 19.5e12 / 1.555e12\n        rows = []\n        for name, (f, b) in c.items():\n            I = float(ns[\"intensity\"](f, b))\n            rows.append([name, f\"{I:.3f}\", \"hesap\" if I > ridge else \"bellek\"])\n        R.table(f\"n = 1024, A100 fp32 (tepe noktası {ridge:.1f} FLOP/bayt)\", [\"işlem\", \"FLOP/bayt\", \"bağımlı\"], rows)\n    R.test(\"op_costs ve sınıflandırma\", oc)\n\n    def mb():\n        rng = np.random.default_rng(0)\n        A, B = rng.normal(size=(12, 8)), rng.normal(size=(8, 20))\n        close(need(ns, \"matmul_blocked\")(A, B, 4), A @ B)\n        close(ns[\"matmul_blocked\"](A, B, 5), A @ B, what=\"T bölmüyor (kenar tile'ları)\")\n    R.test(\"matmul_blocked (dikdörtgen, kenar tile'ları)\", mb)\n",
 "d": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    ext = need(ns, \"ext\", \"derlenmiş modül\")\n\n    def eq():\n        g = torch.Generator().manual_seed(0)\n        x, b = torch.randn(37, 129, generator=g) * 3, torch.randn(129, generator=g)\n        close(ext.bias_gelu(x, b), F.gelu(x + b, approximate=\"tanh\"), atol=1e-5)\n    R.test(\"bias_gelu = F.gelu(x + b, tanh)\", eq)\n\n    def checks():\n        x, b = torch.randn(8, 4), torch.randn(4)\n        bad = [(\"transpoze (bitişik değil)\", lambda: ext.bias_gelu(torch.randn(4, 8).t(), b)),\n               (\"float64\", lambda: ext.bias_gelu(x.double(), b.double())),\n               (\"boyut uyuşmazlığı\", lambda: ext.bias_gelu(x, torch.randn(5)))]\n        for what, fn in bad:\n            try:\n                fn()\n            except RuntimeError:\n                continue\n            raise AssertionError(f\"{what}: TORCH_CHECK hata fırlatmalıydı\")\n    R.test(\"TORCH_CHECK'ler yanlış girdiyi reddediyor\", checks)\n\n    def cuda():\n        if not torch.cuda.is_available():\n            return \"CUDA yok: GPU kısmı atlandı (Colab'da GPU çalışma zamanıyla dene)\"\n        src = ns.get(\"CUDA_SRC\")\n        if not src:\n            return \"CUDA var ama CUDA_SRC boş (isteğe bağlı)\"\n        import hashlib\n        from torch.utils.cpp_extension import load_inline\n        m = load_inline(name=\"ptl_bgc_\" + hashlib.md5(src.encode()).hexdigest()[:8],\n                        cpp_sources=\"torch::Tensor bias_gelu_cuda(torch::Tensor x, torch::Tensor b);\",\n                        cuda_sources=src, functions=[\"bias_gelu_cuda\"], verbose=False)\n        x, b = torch.randn(300, 77, device=\"cuda\"), torch.randn(77, device=\"cuda\")\n        close(m.bias_gelu_cuda(x, b).cpu(), F.gelu(x + b, approximate=\"tanh\").cpu(), atol=1e-5)\n        return \"CUDA çekirdeği doğru\"\n    R.test(\"GPU (isteğe bağlı)\", cuda)\n",
 "e": "import torch\nfrom labcheck import close, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    ext = need(ns, \"ext\", \"derlenmiş modül\")\n    g = torch.Generator().manual_seed(0)\n    A, B = torch.randn(70, 45, generator=g), torch.randn(45, 33, generator=g)\n    R.test(\"matmul_ijk\", lambda: close(ext.matmul_ijk(A, B), A @ B, atol=1e-4))\n    R.test(\"matmul_ikj\", lambda: close(ext.matmul_ikj(A, B), A @ B, atol=1e-4))\n    R.test(\"matmul_tiled (T = 16, kenarlar dahil)\", lambda: close(ext.matmul_tiled(A, B, 16), A @ B, atol=1e-4))\n    n = 512\n    X, Y = torch.randn(n, n, generator=g), torch.randn(n, n, generator=g)\n    rows = []\n    for name, fn in ((\"ijk (B'yi sütun boyunca)\", lambda: ext.matmul_ijk(X, Y)), (\"ikj (ardışık)\", lambda: ext.matmul_ikj(X, Y)),\n                     (\"tiled ikj, T = 64\", lambda: ext.matmul_tiled(X, Y, 64)), (\"torch.matmul (BLAS)\", lambda: X @ Y)):\n        t = bench(fn, repeat=3, number=1)\n        rows.append([name, fmt_time(t), f\"{2 * n ** 3 / t / 1e9:.1f}\"])\n    R.table(f\"n = {n}, tek thread C++ vs BLAS\", [\"yöntem\", \"süre\", \"GFLOP/s\"], rows,\n            note=\"Aynı FLOP, farklı bellek erişim düzeni. BLAS ayrıca SIMD, çok thread ve register tiling kullanır.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m7_1_cuda as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**CUDA yürütme modeli.** Bir çekirdek (kernel) binlerce thread'de aynı anda çalışır. Thread'ler **block**'lara, block'lar **grid**'e gruplanır:

$$
i=\texttt{blockIdx.x}\cdot\texttt{blockDim.x}+\texttt{threadIdx.x}.
$$

Aynı block'taki thread'ler hızlı **shared memory**'yi paylaşır ve `__syncthreads()` ile eşzamanlanır; global bellek (HBM) büyük ama yavaştır.

**tiling.** Naif matmul'de her $C_{ij}$, $A$'nın bir satırını ve $B$'nin bir sütununu global bellekten okur: toplam $2n^3$ okuma. $T\times T$ tile'larla bir block, $A$ ve $B$'nin tile'larını bir kez shared memory'ye yükler ve $T$ kez yeniden kullanır: $2n^3/T$.

**roofline.** Aritmetik yoğunluk $I=\text{FLOP}/\text{bayt}$. Ulaşılabilir performans $\min(\text{tepe FLOP},\ \text{bant genişliği}\times I)$. $I$ tepe noktasının altındaysa işlem **bellek-bağımlı**dır.

**Extension.** `torch.utils.cpp_extension.load_inline` bir C++ (ve isteğe bağlı CUDA) kaynağını derleyip Python modülü olarak yükler; fonksiyonlar `torch::Tensor` alır ve döndürür.

Bu modül GPU olmadan da çalışır: simülasyon (b), model (c) ve C++ (d, e) CPU'da. GPU varsa d aşamasında aynı fonksiyonun CUDA hâli de denenir.

## 1 · Elle

Tiling'de her block, k-ekseni boyunca $n/T=2$ adımda $A$'dan bir $2\times2$ tile, $B$'den bir $2\times2$ tile yükler: block başına $2\cdot(4+4)=16$, dört block'ta 64. Naifin yarısı: kazanç tile boyutu $T$ kadar.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Vektör toplama, $N=1000$, `blockDim.x = 256`**

- `grid`: Gereken block sayısı
- `idle`: Boşta kalan (sınır kontrolüne takılan) thread
- `idx`: `blockIdx.x = 2`, `threadIdx.x = 5` → global indeks

**$4\times4$ matmul $C=AB$, global bellekten okunan eleman sayısı**

- `naive`: Naif: her $C_{ij}$ kendi satır/sütununu okur
- `tiled`: $2\times2$ tile, shared memory ile
- `ai`: fp32 matmul, $n=1024$, ideal aritmetik yoğunluk (FLOP/bayt, 3 matris bir kez taşınır)

In [ ]:
cevaplar = dict(
    grid=None,
    idle=None,
    idx=None,
    naive=None,
    tiled=None,
    ai=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"grid":[4,0],"idle":[24,0],"idx":[517,0],"naive":[128,0],"tiled":[64,0],"ai":[170.666667,0.01]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, CUDA'yı taklit eden bir simülatör. `Ctx` (block/thread indeksleri, `blockDim`) ve okuma sayan `Global` hazır.

- `launch(kernel, grid, block, *args)` — 1-boyutlu: her `b in range(grid)`, `t in range(block)` için `kernel(Ctx(b, t, block), *args)`.
- `vec_add(ctx, x, y, out, n)` — thread başına bir eleman, **sınır kontrolüyle**.
- `matmul_naive(A, B, n)` → `C` (düz liste, satır-ana); `A`, `B` `Global`.
- `matmul_tiled(A, B, n, T)` — her $(b_i, b_j)$ block'u için k-tile'larını yerel ("shared") listelere **bir kez** yükle, sonra o tile'lardan hesapla. $n$, $T$'ye bölünür.

Kontrol okuma sayılarını $2n^3$ ve $2n^3/T$ ile karşılaştırır.

In [ ]:
%%writefile stage_b.py
class Ctx:
    def __init__(self, b, t, dim):
        self.blockIdx, self.threadIdx, self.blockDim = b, t, dim


class Global:
    """Global bellek: her okumayı sayar."""
    def __init__(self, data):
        self.data, self.reads = list(data), 0

    def __getitem__(self, i):
        self.reads += 1
        return self.data[i]

    def __setitem__(self, i, v):
        self.data[i] = v


# ---- senin kodun ----

def launch(kernel, grid, block, *args):
    ...


def vec_add(ctx, x, y, out, n):
    ...


def matmul_naive(A, B, n):
    ...


def matmul_tiled(A, B, n, T):
    ...


n = 1000
x, y, out = Global(range(n)), Global(range(n)), Global([0] * n)
launch(vec_add, (n + 255) // 256, 256, x, y, out, n)
print(out.data[:5], x.reads)                     # [0, 2, 4, 6, 8] 1000
A, B = Global(range(16)), Global(range(16))
print(matmul_tiled(A, B, 4, 2)[:4], A.reads + B.reads)   # 64 okuma

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy.

- `intensity(flops, bytes_)` — eleman eleman (dizi destekli).
- `attainable(peak, bw, I)` → $\min(\text{peak}, \text{bw}\cdot I)$; döngüsüz.
- `op_costs(n)` → `{ad: (flop, bayt)}`, fp32, her tensör bir kez okunup/yazılır:
  - `"add"`: $n$ elemanlı iki vektör toplanır.
  - `"softmax"`: $n\times n$ satır softmax'ı, FLOP $\approx5n^2$, okuma + yazma.
  - `"matmul"`: $n\times n$ iki matris, FLOP $2n^3$, üç matris.
- `matmul_blocked(A, B, T)` — üç seviye tile döngüsü (i, j, k blokları), içte NumPy alt-matris çarpımı.

In [ ]:
%%writefile stage_c.py
import numpy as np


def intensity(flops, bytes_):
    ...


def attainable(peak, bw, I):
    ...


def op_costs(n):
    ...


def matmul_blocked(A, B, T):
    ...


A100 = dict(peak=19.5e12, bw=1.555e12)
for name, (f, b) in op_costs(4096).items():
    I = intensity(f, b)
    print(name, round(I, 3), f"{attainable(A100['peak'], A100['bw'], I) / 1e12:.2f} TFLOP/s")

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

`load_inline` ile C++ operatörü:

- `CPP_SRC` — `torch::Tensor bias_gelu(torch::Tensor x, torch::Tensor b)`: `x` `(n, d)` fp32 bitişik, `b` `(d,)`; $y=\operatorname{gelu}_{\tanh}(x+b)$. `TORCH_CHECK` ile boyut, dtype ve bitişikliği doğrula. Ham işaretçilerle iki döngü.
- `build()` → derlenmiş modül (`load_inline(name=…, cpp_sources=CPP_SRC, functions=["bias_gelu"])`). İlk derleme ~20 sn sürer, sonra önbellekten gelir.

GPU varsa (isteğe bağlı) `CUDA_SRC` içinde aynı işi yapan bir `__global__` çekirdek ve onu çağıran `torch::Tensor bias_gelu_cuda(torch::Tensor x, torch::Tensor b)` yaz (thread başına bir eleman, `blocks = ⌈N/256⌉`). Kontrol yalnız CUDA varken onu `cuda_sources=` ile derleyip sınar.

In [ ]:
%%writefile stage_d.py
import hashlib
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load_inline

CPP_SRC = r"""
#include <torch/extension.h>
#include <cmath>

torch::Tensor bias_gelu(torch::Tensor x, torch::Tensor b) {
    // TODO: TORCH_CHECK'ler, sonra y = gelu_tanh(x + b)
    return x;
}
"""

CUDA_SRC = None   # isteğe bağlı: GPU varsa kendi __global__ çekirdeğin


def build():
    name = "ptl_bias_gelu_" + hashlib.md5(CPP_SRC.encode()).hexdigest()[:8]
    return load_inline(name=name, cpp_sources=CPP_SRC, functions=["bias_gelu"], verbose=False)


ext = build()
x, b = torch.randn(4, 8), torch.randn(8)
print((ext.bias_gelu(x, b) - F.gelu(x + b, approximate="tanh")).abs().max())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

C++'ta üç matmul, aynı derleme biriminde:

- `matmul_ijk(A, B)` — klasik üçlü döngü, en içte $k$: $B$'yi sütun boyunca (stride $n$) okur.
- `matmul_ikj(A, B)` — en içte $j$: hem $B$'nin satırı hem $C$'nin satırı ardışık okunur/yazılır (önbellek dostu, derleyici vektörleştirebilir).
- `matmul_tiled(A, B, T)` — $i$, $k$, $j$ tile'ları + içte ikj.

Kontrol doğruluğu sınar ve `torch.matmul` (BLAS) ile süreleri karşılaştırır.

In [ ]:
%%writefile stage_e.py
import hashlib
import torch
from torch.utils.cpp_extension import load_inline

SRC = r"""
#include <torch/extension.h>

torch::Tensor matmul_ijk(torch::Tensor A, torch::Tensor B) {
    // TODO
    return A;
}

torch::Tensor matmul_ikj(torch::Tensor A, torch::Tensor B) {
    // TODO
    return A;
}

torch::Tensor matmul_tiled(torch::Tensor A, torch::Tensor B, int64_t T) {
    // TODO
    return A;
}
"""


def build():
    name = "ptl_mm_" + hashlib.md5(SRC.encode()).hexdigest()[:8]
    return load_inline(name=name, cpp_sources=SRC, functions=["matmul_ijk", "matmul_ikj", "matmul_tiled"],
                       extra_cflags=["-O3"], verbose=False)


ext = build()
A, B = torch.randn(64, 64), torch.randn(64, 64)
print((ext.matmul_ikj(A, B) - A @ B).abs().max())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Williams, Waterman, Patterson (2009): Roofline modeli

[Williams, Waterman, Patterson — Roofline: An Insightful Visual Performance Model for Multicore Architectures (CACM 2009)](https://dl.acm.org/doi/10.1145/1498765.1498785)

Kendi makinenin (CPU ve varsa GPU) roofline'ını çıkar: STREAM benzeri bir triad çekirdeğiyle tepe bant genişliğini, yoğun matmul ile tepe FLOP/s'yi ölç. Makale Şekil 2'deki gibi çiz ve c aşamasındaki işlemleri (vektör toplama, softmax, matmul, konvolüsyon) ölçülen aritmetik yoğunluk ve elde edilen performansla noktalar halinde yerleştir; hangilerinin tavanların altında kaldığını ve nedenini (önbellek, vektörleştirme, paralellik) açıkla.


**Adımlar**

1. Bant genişliği için dizi boyutunu son düzey önbellekten büyük seç.

**Kabul ölçütleri**

- Ölçülen tavanlar donanım özelliklerinin makul bir yüzdesinde.
- Her işlem için 'hangi tavan' gerekçesi.

## Makale ödevi: Volkov & Demmel (2008): GPU'da yoğun lineer cebir

[Volkov & Demmel — Benchmarking GPUs to Tune Dense Linear Algebra (SC 2008)](https://dl.acm.org/doi/10.5555/1413370.1413402)

b aşamasındaki tiled matmul simülatörünü bir CUDA (ya da Triton, M7.2) çekirdeğine çevir; paylaşımlı bellek tiling'i, kayıt (register) bloklaması ve thread başına birden çok çıktı ile adım adım hızlan; her adımda global bellek trafiğini (simülatörün okuma sayacıyla) ve ölçülen GFLOP/s'yi karşılaştır. Makalenin "daha az thread, daha çok iş" gözlemini doluluk (occupancy) ölçerek test et.


**Adımlar**

1. GPU yoksa Triton'u TRITON_INTERPRET=1 ile CPU'da doğrula, performansı Colab'da ölç.

**Kabul ölçütleri**

- Her optimizasyon adımında ölçülen hızlanma ve trafik azalması tablosu.
- Düşük doluluk + yüksek ILP'nin yüksek doluluktan hızlı olabildiği bir örnek.

## Kaynaklar

- [GPU MODE — Lectures (CUDA, profiling, tiling)](https://github.com/gpu-mode/lectures) — Apache-2.0
- [Hwu, Kirk, El Hajj — Programming Massively Parallel Processors (4. baskı)](https://www.elsevier.com/books/programming-massively-parallel-processors/hwu/978-0-323-91231-0) — ©
- [PyTorch Tutorials — Custom C++ and CUDA Extensions](https://docs.pytorch.org/tutorials/advanced/cpp_extension.html) — BSD-3-Clause
- [Williams, Waterman, Patterson — Roofline: An Insightful Visual Performance Model](https://dl.acm.org/doi/10.1145/1498765.1498785) — ©
- [MiniTorch — Module 3: Efficiency (CUDA)](https://minitorch.github.io/) — lisans dosyası yok
- [Simon Boehm — How to Optimize a CUDA Matmul Kernel](https://siboehm.com/articles/22/CUDA-MMM) — ©